# 01C - Cinematic Piano Memory

**Guiding question:** with the same clock but a different opening key cue, will the reader change the ending of the phrase?

> **What you finished last time:** [01B's notebook](01B-sequence-memory-pytorch-lab.ipynb) kept the final real state, skipped blank answers, and checked future access.
> **What this notebook delivers:** one authored phrase in two keys, fitted RNN/LSTM readers, actual prediction audio, and a direct opening-cue intervention.
> **What this chapter is not:** a composer, a generalization benchmark, or a fair-parameter-count architecture contest.

We keep the familiar **D3 A3 D4** opening, extend it to a 24-note phrase,
and make a second version **three semitones higher**. The model gets the key
cue once, at the start. After that, both versions have exactly the same inputs.

The earlier two-number notecard is now a learned hidden row. Does it carry
enough of that opening difference to produce two different endings?


In [ ]:
# -- Setup: reuse the canonical sequence helpers ---------------------------------
import io
import wave
import numpy as np
import torch
from torch import nn
import matplotlib.pyplot as plt
from IPython.display import Audio, display

from sequence_helpers import backward_step, seed_everything, trainable_parameter_count

seed_everything(17)
torch.set_num_threads(1)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
plt.rcParams.update({"figure.facecolor": "#fffaf1", "axes.facecolor": "#fffaf1"})
print(f"Device: {device}")


## 1 - Make the clock unable to answer the question alone

First hear the authored D-minor phrase. Then imagine playing the same
phrase in F minor, shifting every pitch up by three semitones. Its timing
stays exactly the same, but its ending must be higher too.

![The D-minor version begins with a D3 cue and ends on D4; the F-minor version begins with F3 and ends on F4. Both have the same repeating clock and no further pitch cues.](images/01c-01-same-clock-different-key.png)

*Compare the middle cards: they match. A clock-only reader couldn't choose
different endings. The opening cue is the only input difference it can carry.*

We use two clock features, sine and cosine, which repeat every six notes.
A third feature contains the opening pitch **only at step zero** and zeros
afterward. We encode it relative to a fixed reference pitch; both opening
cues are nonzero and different.
Unlike 01B's note IDs, this cue is a measured pitch feature, not a lookup
address. We are predicting continuous pitches here, not choosing vocabulary IDs.

**Why two versions?** One fixed phrase could be memorized without using a
key cue at all. Here identical later clocks demand different later pitches.
We can verify the input difference, then swap only that cue and measure
the model's response. These are still practiced examples, not unseen-key tests.

**Predict:** if we remove both opening cues, can a deterministic reader
produce both correct endings from identical remaining inputs?


In [ ]:
# -- Original D-minor progression and lightweight piano synthesizer --------------
midi_notes = np.array([
    50, 57, 62, 65, 69, 65,
    46, 53, 58, 62, 65, 62,
    41, 48, 53, 57, 60, 57,
    48, 55, 60, 64, 67, 62,
], dtype=np.float32)
# Broadcast the same pitch offset/scale over [24] notes; shape stays [24], now measured in octaves.
normalized_target = torch.tensor((midi_notes - 50.0) / 12.0, dtype=torch.float32)

def synthesize_piano(midi, seconds_per_note=0.16, sample_rate=16000):
    chunks = []
    for note in midi:
        t = np.linspace(0, seconds_per_note, int(sample_rate * seconds_per_note), endpoint=False)
        frequency = 440.0 * 2.0 ** ((float(note) - 69.0) / 12.0)
        envelope = np.exp(-5.5 * t)
        tone = (
            np.sin(2 * np.pi * frequency * t)
            + 0.35 * np.sin(2 * np.pi * 2 * frequency * t)
            + 0.12 * np.sin(2 * np.pi * 3 * frequency * t)
        ) * envelope
        chunks.append(tone)
    # Concatenate note waveforms [S] end to end -> [notes*S], not a new note/channel axis.
    audio = np.concatenate(chunks)
    audio = 0.85 * audio / max(np.abs(audio).max(), 1e-8)
    return audio.astype(np.float32), sample_rate

target_audio, sample_rate = synthesize_piano(midi_notes)
print(f"Phrase length: {len(midi_notes)} notes")
print("This is the authored D-minor target; the next cell makes its transposed partner.")
display(Audio(target_audio, rate=sample_rate))


In [ ]:
# -- Two keys, identical later clocks, different opening cues ---------------------
steps = torch.arange(len(midi_notes), dtype=torch.float32)
phase = 2 * torch.pi * steps / 6
clock = torch.stack([torch.sin(phase), torch.cos(phase), torch.zeros_like(steps)], dim=-1)
transpose_semitones = torch.tensor([0.0, 3.0])
base_phrase = 0
features = clock.unsqueeze(0).repeat(len(transpose_semitones), 1, 1)
features[:, 0, 2] = (float(midi_notes[0]) + transpose_semitones - 48.0) / 12.0
targets = normalized_target[None, :, None] + transpose_semitones[:, None, None] / 12.0

assert features.shape == (2, 24, 3) and targets.shape == (2, 24, 1)
assert torch.equal(features[0, 1:], features[1, 1:])
assert torch.count_nonzero(features[:, 1:, 2]) == 0
assert features[0, 0, 2] != features[1, 0, 2] and bool((features[:, 0, 2] != 0).all())
assert torch.allclose(targets[1] - targets[0], torch.full_like(targets[0], 3.0 / 12.0))
print("Features:", tuple(features.shape), "= (two phrases, 24 steps, three features)")
print("Opening pitch-cue values:", features[:, 0, 2].tolist())
print("Later input rows match exactly; target pitches differ by three semitones.")
print("Takeaway: unlike one fixed phrase, this pair makes ignoring the opening cue insufficient.")


## 2 - Same practice task, two different readers

Train a vanilla RNN and an LSTM on **both** key versions. Each makes one
pitch guess at every step. The pitch head, hidden width, optimizer, epoch
budget, and clipping threshold have the same settings.

| Part | RNN | LSTM |
|---|---|---|
| Input | Same two key cues and repeating clocks | Same |
| Carried state | One hidden row | Cell state plus exposed hidden row |
| Prediction head | Hidden row to one pitch | Same kind of head, independently trained |
| Training | Fit the two practiced pitch paths | Same |

This is **not** equal parameter count. LSTM gates add weights; the printed
counts make that cost visible. Nor do matching seeds give different
architectures identical initial tensors. We compare these two training
setups, not prove that gates alone caused any performance gap.

Read the wrapper as three jobs: choose the recurrent cell, turn every
hidden row into a pitch, then use the ordinary training helper. Saved
checkpoints contain actual predictions for the D-minor version, not a
hand-drawn improvement curve.

**Predict:** can both readers lower training loss? Must the LSTM win?
Keep the printed results even if they disagree with your prediction.


In [ ]:
# -- Transparent model wrapper: the cell type is the controlled variable --------
class PianoMemoryModel(nn.Module):
    def __init__(self, cell_type, hidden_size=24):
        super().__init__()
        recurrent_class = nn.RNN if cell_type == "rnn" else nn.LSTM
        self.recurrent = recurrent_class(3, hidden_size, batch_first=True)
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, sequence):
        # [B,T,3] -> [B,T,H]; the shared H -> 1 head predicts pitch at every timestep.
        hidden_path, _ = self.recurrent(sequence)
        return self.head(hidden_path), hidden_path

def train_model(cell_type, hidden_size=24, epochs=500, snapshot_epochs=(0, 50, 150, 499)):
    seed_everything(17)
    model = PianoMemoryModel(cell_type, hidden_size).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.02)
    snapshots = {}
    losses = []
    norms = []

    for epoch in range(epochs):
        predictions, _ = model(features.to(device))
        loss = nn.functional.mse_loss(predictions, targets.to(device))
        pre_clip, post_clip = backward_step(loss, model, optimizer, clip_norm=1.0)
        losses.append(float(loss.detach()))
        norms.append((pre_clip, post_clip))
        if epoch in snapshot_epochs:
            # Save the D-minor row only: [2,24,1] -> [24]; never squeeze the two phrases together.
            snapshots[epoch] = predictions[base_phrase, :, 0].detach().cpu().numpy()
    return model, losses, norms, snapshots

rnn_model, rnn_losses, rnn_norms, rnn_snapshots = train_model("rnn")
lstm_model, lstm_losses, lstm_norms, lstm_snapshots = train_model("lstm")
print(f"RNN parameters:  {trainable_parameter_count(rnn_model):,}")
print(f"LSTM parameters: {trainable_parameter_count(lstm_model):,}")
print(f"Final practice MSE - RNN: {rnn_losses[-1]:.6f}, LSTM: {lstm_losses[-1]:.6f}")
assert all(np.isfinite(history).all() and history[-1] < history[0] for history in (rnn_losses, lstm_losses))
print("Both losses are practice scores on the same two phrases, not held-out quality scores.")


In [ ]:
# -- Visual payoff: learning curves and final pitch paths -------------------------
with torch.inference_mode():
    rnn_prediction, _ = rnn_model(features.to(device))
    lstm_prediction, _ = lstm_model(features.to(device))
# Plot the D-minor row: [2,24,1] -> [24]; convert normalized pitches back to MIDI.
rnn_midi = rnn_prediction[base_phrase, :, 0].cpu().numpy() * 12 + 50
lstm_midi = lstm_prediction[base_phrase, :, 0].cpu().numpy() * 12 + 50

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].semilogy(rnn_losses, label="RNN", color="#f97316")
axes[0].semilogy(lstm_losses, label="LSTM", color="#60a5fa")
axes[0].set(title="Matched training", xlabel="Epoch", ylabel="MSE")
axes[0].legend()
axes[1].plot(midi_notes, label="target", color="#34d399", linewidth=3)
axes[1].plot(rnn_midi, label="RNN", color="#f97316")
axes[1].plot(lstm_midi, label="LSTM", color="#60a5fa")
axes[1].set(title="Final pitch path", xlabel="Timestep", ylabel="MIDI note")
axes[1].legend()
plt.show()


### The curve fell. Did the opening cue actually matter?

A low practice error alone isn't enough. Swap just the opening cue of the
D-minor input for the F-minor cue; leave every later clock value unchanged.
Compare the final predicted pitches. The correct change is **+3 semitones**.

Also erase both cues. Those inputs now match at **every** position, so a
deterministic model must produce identical pitch paths for both rows.
Their correct endings differ by three semitones: no identical prediction
can hit both. Even the best compromise has at least 1.5 semitones of ending
RMSE across this pair.

**Predict:** should swapping the cue change only the first prediction, or
can the change still reach the ending? Does hearing one good phrase answer that?


In [ ]:
# -- Swap only the first pitch cue; measure the last predicted pitch --------------
cue_swap = features[base_phrase:base_phrase + 1].clone()
cue_swap[0, 0, 2] = features[1, 0, 2]
assert torch.equal(cue_swap[0], features[1])
erased_cues = features.clone()
erased_cues[:, 0, 2] = 0
assert torch.equal(erased_cues[0], erased_cues[1])

for name, fitted in [("RNN", rnn_model), ("LSTM", lstm_model)]:
    fitted.eval()
    with torch.inference_mode():
        original, _ = fitted(features[base_phrase:base_phrase + 1].to(device))
        swapped, _ = fitted(cue_swap.to(device))
        erased, _ = fitted(erased_cues.to(device))
    ending_shift = ((swapped[0, -1, 0] - original[0, -1, 0]) * 12).item()
    assert torch.allclose(erased[0], erased[1], atol=1e-6)
    print(f"{name}: opening-only cue swap changed the ending by {ending_shift:+.3f} semitones; expected +3.")
    if abs(ending_shift - 3.0) < 0.5:
        print("  This practiced cue difference survived to the ending within half a semitone.")
    else:
        print("  This run did not preserve the ending shift within half a semitone; keep that limitation.")
minimum_cueless_rmse = transpose_semitones.std(unbiased=False).item()
assert minimum_cueless_rmse == 1.5
print(f"Identical cue-erased inputs cannot fit both endings: best possible ending RMSE >= {minimum_cueless_rmse:.1f} semitones.")


On the authoring CPU run, the ending moved **+2.944 semitones for the RNN**
and **+2.908 for the LSTM**, close to the practiced +3 change. Both readers
carried a useful difference across the phrase. The RNN also had lower overall
practice loss in that run: this isn't an experiment arranged to make LSTM win.

Read your printed result if library or hardware versions change it. Matching
two practiced keys doesn't prove the model learned a smooth transposition rule
for keys in between or outside them. We'll inspect that boundary with a local
sensitivity measurement after listening.


## 3 - Hear checkpointed practice

Listen for the broad pitch path approaching the authored target. The player
turns saved predictions into simple synthesized tones; it isn't a recording
of a trained instrument model.

Each saved array is the prediction made **before** that numbered update.
Epoch zero is the actual random start. The curves measure both key versions;
these checkpoint players play just the D-minor one so the comparison is audible.
Good sound on one practiced phrase is not evidence about new music.


In [ ]:
# -- Audible checkpoints from actual saved predictions ---------------------------
for epoch in sorted(lstm_snapshots):
    checkpoint_midi = lstm_snapshots[epoch] * 12 + 50
    audio, rate = synthesize_piano(checkpoint_midi)
    print(f"LSTM checkpoint epoch {epoch}, MSE={lstm_losses[epoch]:.6f}")
    display(Audio(audio, rate=rate))


## 4 - A tiny gradient might mean two different things

We already changed the cue and measured the ending. Now inspect a **local**
feedback route near the current cue value.

If the ending is almost correct, its loss can send only a tiny correction.
That doesn't mean memory failed. If the ending is wrong **and** the cue gets
almost no feedback, the training route may be hard to use.

Report three things together:

- **Ending error:** how wrong are the final pitches on the two practiced keys?
- **Loss gradient to the cue:** how strongly does that error teach the first pitch-cue feature?
- **Ending sensitivity:** how does the final prediction respond locally to that first cue,
  before loss size scales the feedback? For perfect transposition here, the slope would be +1.

Only feature 2 is the pitch cue; gradients to the two clock features are not
cue gradients. A local derivative is also not the same as the three-semitone
intervention we just ran.

The input-gradient probe is a diagnostic on a copy of the numerical features.
It asks how the prediction depends on a cue; it doesn't add the data to the
optimizer or change the stored training inputs. Fixed note IDs in 01B aren't
differentiated or adjusted this way.

**Predict:** can a nearly correct model have a small loss gradient and a
nonzero ending sensitivity at the same time?


In [ ]:
# -- Separate error-scaled feedback from the local cue-to-ending route ------------
def opening_cue_gradient(model):
    probe = features.clone().to(device).requires_grad_(True)
    prediction, _ = model(probe)
    final_loss = (prediction[:, -1] - targets.to(device)[:, -1]).pow(2).mean()
    loss_gradient = torch.autograd.grad(final_loss, probe, retain_graph=True)[0][:, 0, 2]
    sensitivity = torch.autograd.grad(prediction[:, -1, 0].sum(), probe)[0][:, 0, 2]
    return (
        float(loss_gradient.abs().mean()), float(final_loss.detach()),
        float(sensitivity.mean()),
    )

for name, fitted in [("RNN", rnn_model), ("LSTM", lstm_model)]:
    cue_feedback, ending_error, ending_slope = opening_cue_gradient(fitted)
    print(f"{name}: ending MSE={ending_error:.8f}; mean absolute cue-loss gradient={cue_feedback:.8f}")
    print(f"  Mean local cue-to-ending slope: {ending_slope:+.5f}; ideal transposition slope: +1.")
print("Takeaway: interpret feedback with the error that caused it; don't equate a small gradient with forgetting.")


Here's the catch: the authoring run's local slopes were only about **+0.26**
and **+0.45**, not the ideal +1. The two practiced endpoints were nearly right,
but small cue changes around them didn't produce a perfectly proportional
pitch change. Hitting two stepping stones isn't the same as paving the path
between them. Keep that distinction before calling this general musical memory.


## 5 - Your turn: give the reader a different-sized card

Change only `hidden_size_to_test`. The training pair, epoch budget, optimizer,
and clipping stay fixed. A wider card can store more, but also adds weights
and changes optimization; it needn't improve this run.

**Your turn:** predict whether a 12-number LSTM card fits these two phrases
as well as the original 24-number card. Compare both error and parameter cost.


In [ ]:
# -- Your turn: one controlled hidden-size comparison ----------------------------
# CHANGE THIS: try 12, 24, or 36
hidden_size_to_test = 12
test_model, test_losses, _, _ = train_model(
    "lstm",
    hidden_size=hidden_size_to_test,
    epochs=500,
    snapshot_epochs=(),
)
print(f"Hidden size {hidden_size_to_test}: {trainable_parameter_count(test_model):,} parameters")
print(f"Final 500-epoch MSE: {test_losses[-1]:.6f}")
print(f"Original 24-number LSTM, same 500 epochs: {lstm_losses[-1]:.6f}")
print("Check: compare both error and parameter count; width is not free.")


In [ ]:
# -- Final inline players: authored target, RNN, and LSTM -------------------------
for label, notes in [
    ("Authored target", midi_notes),
    ("Vanilla RNN", rnn_midi),
    ("LSTM", lstm_midi),
]:
    audio, rate = synthesize_piano(notes)
    print(label)
    display(Audio(audio, rate=rate))


### What changed from the little notecard?

In 01A the reader's numbers and gates were chosen by hand. Here training
adjusted the recurrent weights and pitch head together. The state is still
a short row carried from one step to the next, not a saved transcript.

The two-key cue intervention tests whether an input difference reaches the
ending. It does not isolate a named memory coordinate, prove new-key
generalization, or establish that every LSTM beats every RNN.

**Forward:** [02A's notebook](../02-tokenization-and-embeddings/02A-tokenization-and-embeddings-theory.ipynb)
returns to how text becomes IDs and row inputs. Later,
[03C's notebook](../03-language-models/03C-fixed-windows-to-recurrent-language-models.ipynb)
reuses this running-memory idea for next-word prediction. The data changes;
the carried-card question doesn't.


## 6 - Evidence scorecard

| Question | Evidence | Limit |
|---|---|---|
| Can the clock alone select the key? | Later inputs match; desired endings differ by three semitones | A single fixed phrase would not establish this |
| Did the opening cue affect the ending? | Opening-only intervention prints the final semitone shift | Read the actual result, including any miss |
| Did we make a fair architecture ranking? | Same hidden width and training settings; different parameter counts | No; capacity and optimization differ |
| Are the sounds actual predictions? | Saved pre-update checkpoints and final model outputs drive the players | Listening is practice-fit evidence, not a music benchmark |
| Can a small gradient mean success? | Error-scaled cue gradient and loss-independent local slope are printed separately | Local sensitivity isn't a full memory explanation |

| Tier | What we covered |
|---|---|
| Built and measured | Two-key input/target contract, actual RNN/LSTM fitting, cue swap, cue-erasure impossibility check, saved audio, cue-only derivatives, matched-budget width drill |
| Explained or illustrated | Why cue variation is needed, what a repeated clock cannot reveal, state versus weights, parameter-count confounding |
| Named and out of scope | Unseen-key generalization, polyphonic event tokenization, sustain pedals, autonomous composition, and attention |

**Watch for:** fitting one phrase and calling it key memory; pooling clock
and key gradients; interpreting smaller loss gradients without checking error;
comparing a 250-epoch drill with a 500-epoch baseline.

**Remember:** if the opening cue is the only difference in the input, changing
only that cue is the cleanest first question to ask about the ending.
